In [1]:
import warnings
from pathlib import Path

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
import os

from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=1)

from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


In [4]:
from llama_index.core import Settings
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.openrouter import OpenRouter

# Set project and organization IDs
project_id = "82bf4329-2778-4298-95cc-537e04dabc31"
organization_id = "bb2e5011-f86d-419a-9502-6e5c74849f40"

# set LLM, embedding model
embed_model = OllamaEmbedding(model_name="mxbai-embed-large:latest")
llm = OpenRouter(
    model="meta-llama/llama-4-scout",
    temperature=0.0,
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),
)

# Global settings
Settings.llm = llm
Settings.embed_model = embed_model

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


### Create Splitter

- The document is a consolidated regulatory filing that contains multiple sub-entities - the 7 different Asset Manager funds - with identical reporting structures.

- Our first task is to create a document splitter that can identify the specific page numbers corresponding to each fund.

- This is a pre-requisite for the downstream task of doing per-fund extraction and consolidation

In [6]:
from llama_cloud_services import LlamaParse

parser = LlamaParse(
    api_key=settings.LLAMA_CLOUD_API_KEY.get_secret_value(),
    premium_mode=True,
    result_type="markdown",
    project_id=project_id,
    organization_id=organization_id,
)
result = await parser.aparse("../data/fidelity_fund.pdf")
markdown_nodes = await result.aget_markdown_nodes(split_by_page=True)

Started parsing the file under job_id c64e6150-16cf-4dd9-92d0-3c249d96e991
..

In [7]:
markdown_nodes

[TextNode(id_='c8479b81-2216-48d3-960d-d8b8d1aeba9e', embedding=None, metadata={'page_number': 1, 'file_name': '../data/fidelity_fund.pdf'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='\n5/31/25, 9:52 AM Print Page\n\n# Fidelity Asset Manager® Funds\n\nFidelity Asset Manager® 20%\nFidelity Asset Manager® 30%\nFidelity Asset Manager® 40%\nFidelity Asset Manager® 50%\nFidelity Asset Manager® 60%\nFidelity Asset Manager® 70%\nFidelity Asset Manager® 85%\n\n## Annual Report\n### September 30, 2024\nIncludes Fidelity and Fidelity Advisor share classes\n\n[A stylized graphic of a partial circle with radiating lines, resembling a sunrise or burst, is displayed in gray.]\n\nFidelity\nINVESTMENTS®\n\nabout:blank 1/120', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'),
 TextNode(id_='03de6c67-486c-44ef-8980-6

In [8]:
import pickle

with open("../storage/fidelity_fund/markdown_nodes.pkl", "wb") as f:
    pickle.dump(markdown_nodes, f)
print("Data saved successfully!!!")

Data saved successfully!!!


In [14]:
with open("../storage/fidelity_fund/markdown_nodes.pkl", "rb") as f:
    markdown_nodes_loaded = pickle.load(f)

markdown_nodes_loaded[:2]

[TextNode(id_='c8479b81-2216-48d3-960d-d8b8d1aeba9e', embedding=None, metadata={'page_number': 1, 'file_name': '../data/fidelity_fund.pdf'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='\n5/31/25, 9:52 AM Print Page\n\n# Fidelity Asset Manager® Funds\n\nFidelity Asset Manager® 20%\nFidelity Asset Manager® 30%\nFidelity Asset Manager® 40%\nFidelity Asset Manager® 50%\nFidelity Asset Manager® 60%\nFidelity Asset Manager® 70%\nFidelity Asset Manager® 85%\n\n## Annual Report\n### September 30, 2024\nIncludes Fidelity and Fidelity Advisor share classes\n\n[A stylized graphic of a partial circle with radiating lines, resembling a sunrise or burst, is displayed in gray.]\n\nFidelity\nINVESTMENTS®\n\nabout:blank 1/120', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'),
 TextNode(id_='03de6c67-486c-44ef-8980-6

### Define Splitting Functions

- We then define a set of functions to find the "splits".

- We first identify the split categories given a user description. (e.g. what are the precise labels we want to split on?)
- We then split the document according to the split categories. On (2), we do this by using LLMs on each page to detect if a split starts on that page.
- We then count all pages from the start of the current split to the start on the next split as within the current split.

In [ ]:
from llama_index.core.llms import LLM
from llama_index.core.prompts import ChatMessage, ChatPromptTemplate
from llama_index.core.schema import TextNode
from pydantic import BaseModel

split_category_prompt: str = """\
You are an AI document assistant tasked with finding the 'split categories' given a user description and the \
document text.
- The split categories is a list of string tags from the document that correspond to the user description.
- Do not make up split categories. 
- Do not include category tags that don't fit the user description,\
for instance subcategories or extraneous titles.
- Do not exclude category tags that do fit the user description. 

For instance, if the user asks to "find all top-level sections of an ArXiv paper", then a sample output would be:
["1. Introduction", "2. Related Work", "3. Methodology", "4. Experiments", "5. Conclusion"]

The split description and document text are given below. 

Split description:
{split_description}

Here is the document text:
{document_text}

"""


class SplitCategories(BaseModel):
    """A list of all split categories from a document."""

    split_categories: list[str]


async def afind_split_categories(
    split_description: str,
    nodes: list[TextNode],
    llm: LLM | None = None,
    page_limit: int | None = 5,
) -> list[str]:
    """Find split categories given a user description and the page limit.

    These categories will then be used to find the exact splits of the document. 

    NOTE: with the page limit there is an assumption that the categories are found in the first few pages,\
    for instance in the table of contents. This does not account for the case where the categories are \
    found throughout the document. 

    """
    llm = llm or Settings.llm

    chat_template = ChatPromptTemplate(
        [
            ChatMessage.from_str(split_category_prompt, "user"),
        ]
    )
    nodes_head = nodes[:page_limit] if page_limit is not None else nodes
    doc_text = "\n-----\n".join([n.get_content(metadata_mode="all") for n in nodes_head])

    result = await llm.astructured_predict(
        SplitCategories,
        chat_template,
        split_description=split_description,
        document_text=doc_text,
    )
    return result.split_categories

In [ ]:
description: str = "Find and split by the main funds in this document, should be listed in the first few pages"

split_categories = await afind_split_categories(
    split_description=description,
    nodes=markdown_nodes,
    llm=llm,
    page_limit=5,
)
split_categories

['Fidelity Asset Manager®20%',
 'Fidelity Asset Manager®30%',
 'Fidelity Asset Manager®40%',
 'Fidelity Asset Manager®50%',
 'Fidelity Asset Manager®60%',
 'Fidelity Asset Manager®70%',
 'Fidelity Asset Manager®85%']

### Split Document based on Existing Categories/Rules
